# Лабораторная работа №3

# Проверка статистических гипотез

**Цель работы:**
Проверка гипотезы о нормальном распределении возраста и среднего возраста преступников в Москве в 2021 году, а также гипотез о равенстве дисперсий при уровне значимости α=0.05.
Провела статистический анализ данных с использованием Python. Построила интервальные ряды, рассчитала критерии Пирсона для нормальности и Фишера для дисперсий, проверила значимость гипотез и интерпретировала результаты.

In [1]:
# Импорт библиотек
import math
import numpy as np
import scipy.stats as stats

In [2]:
# Загрузка данных
with open('Москва_2021.txt', 'r') as file:
    data = [int(line.strip()) for line in file if line.strip()]
N = len(data)

### Предварительный анализ данных
Рассчитаем основные характеристики генеральной совокупности.

In [3]:
def mean_calc(data):
    return sum(data) / len(data)

def std_dev_calc(data):
    mean_val = mean_calc(data)
    variance = sum((x - mean_val) ** 2 for x in data) / len(data)
    return variance ** 0.5

gen_mean = mean_calc(data)
gen_std = std_dev_calc(data)
gen_variance = gen_std ** 2
print(f"Генеральное среднее: {gen_mean:.2f}")
print(f"Генеральное СКО: {gen_std:.2f}")
print(f"Генеральная дисперсия: {gen_variance:.2f}")

Генеральное среднее: 35.37
Генеральное СКО: 12.04
Генеральная дисперсия: 144.92


Генеральное среднее ≈35.37 лет, СКО ≈12.04, дисперсия ≈144.92, что указывает на значительный разброс и возможную асимметрию

## Подготовка и первичная обработка данных

### Создание интервального ряда для возраста

In [4]:
def create_interval_series(data, interval_width):
    min_val = math.floor(min(data))
    max_val = math.ceil(max(data))
    num_intervals = math.ceil((max_val - min_val) / interval_width)
    intervals = []
    current = min_val
    for i in range(num_intervals):
        if i == num_intervals - 1:
            intervals.append((current, current + interval_width))
        else:
            intervals.append((current, current + interval_width))
        current += interval_width
    frequencies = [0] * num_intervals
    for value in data:
        for i, (start, end) in enumerate(intervals):
            if i == num_intervals - 1:
                if start <= value <= end:
                    frequencies[i] += 1
                    break
            else:
                if start <= value < end:
                    frequencies[i] += 1
                    break
    return intervals, frequencies

intervals_age, frequencies_age = create_interval_series(data, interval_width=9)

print("Интервальный ряд для возраста:")
for (start, end), freq in zip(intervals_age, frequencies_age):
    print(f"[{start}, {end}): {freq}" if end != intervals_age[-1][1] else f"[{start}, {end}]: {freq}")

Интервальный ряд для возраста:
[14, 23): 4811
[23, 32): 9476
[32, 41): 7243
[41, 50): 7951
[50, 59): 1140
[59, 68): 1472
[68, 77]: 330


Интервальный ряд показывает пиковую частоту в интервале 23-32

### Генерация выборок средних возрастов

In [5]:
def generate_sample_means(data, num_samples=36, confidence_level=0.95, delta=3):
    t = stats.norm.ppf(1 - (1 - confidence_level) / 2)
    sigma = std_dev_calc(data)
    n = math.ceil((t ** 2 * sigma ** 2) / delta ** 2)
    samples = []
    for _ in range(num_samples):
        sample = np.random.choice(a=data, size=n, replace=True)
        samples.append(sample)
    means_of_samples = [np.mean(sample) for sample in samples]
    return means_of_samples

sample_means = generate_sample_means(data)
intervals_means, frequencies_means = create_interval_series(sample_means, interval_width=1)

print("Интервальный ряд для средних возрастов:")
for (start, end), freq in zip(intervals_means, frequencies_means):
    print(f"[{start}, {end}): {freq}" if end != intervals_means[-1][1] else f"[{start}, {end}]: {freq}")

Интервальный ряд для средних возрастов:
[31, 32): 1
[32, 33): 0
[33, 34): 5
[34, 35): 11
[35, 36): 7
[36, 37): 6
[37, 38): 4
[38, 39): 1
[39, 40]: 1


## Статистический анализ

### Проверка гипотезы о нормальном распределении (критерий Пирсона)

Проверим нормальность для возраста и средних возрастов при α=0.05.

In [6]:
def calculate_chi_square(intervals, frequencies, sample_mean, sample_std):
    n = sum(frequencies)
    s = len(intervals)
    theoretical_frequencies = []
    chi_square_components = []
    for i, (start, end) in enumerate(intervals):
        z_start = -np.inf if i == 0 else (start - sample_mean) / sample_std
        z_end = np.inf if i == s - 1 else (end - sample_mean) / sample_std
        p_i = stats.norm.cdf(z_end) - stats.norm.cdf(z_start)
        n_i_theoretical = n * p_i
        theoretical_frequencies.append(n_i_theoretical)
        if n_i_theoretical > 0:
            component = ((frequencies[i] - n_i_theoretical) ** 2) / n_i_theoretical
            chi_square_components.append(component)
        else:
            chi_square_components.append(0)
    chi_square_observed = sum(chi_square_components)
    return chi_square_observed, theoretical_frequencies

def pearson_chi_square_test(intervals, frequencies, alpha=0.05):
    n = sum(frequencies)
    midpoints = [(start + end) / 2 for start, end in intervals]
    sample_mean = sum(midpoints[i] * frequencies[i] for i in range(len(midpoints))) / n
    variance = sum(frequencies[i] * (midpoints[i] - sample_mean) ** 2 for i in range(len(midpoints))) / n
    sample_std = math.sqrt(variance)
    chi_square_observed, theoretical_frequencies = calculate_chi_square(intervals, frequencies, sample_mean, sample_std)
    s = len(intervals)
    r = 2  # параметры нормального распределения
    degrees_of_freedom = s - 1 - r
    chi_square_critical = stats.chi2.ppf(1 - alpha, degrees_of_freedom)
    print("\nИнтервалы и частоты:")
    print("Интервал\t\tЭмпир. частота\tТеор. частота")
    for i, (start, end) in enumerate(intervals):
        interval_str = f"[{start}, {end})" if i < len(intervals) - 1 else f"[{start}, {end}]"
        print(f"{interval_str}\t\t{frequencies[i]}\t\t{theoretical_frequencies[i]:.2f}")
    print(f"Наблюдаемое значение χ²: {chi_square_observed:.4f}")
    print(f"Критическое значение χ²: {chi_square_critical:.4f}")
    if chi_square_observed > chi_square_critical:
        print("Нулевая гипотеза отвергнута: распределение не нормальное")
    else:
        print("Нет оснований отвергнуть гипотезу: распределение нормальное")

print("\nПроверка для возраста:")
pearson_chi_square_test(intervals_age, frequencies_age)

print("\nПроверка для средних возрастов:")
pearson_chi_square_test(intervals_means, frequencies_means)


Проверка для возраста:

Интервалы и частоты:
Интервал		Эмпир. частота	Теор. частота
[14, 23)		4811		4863.87
[23, 32)		9476		7550.33
[32, 41)		7243		9316.57
[41, 50)		7951		6827.43
[50, 59)		1140		2970.25
[59, 68)		1472		766.32
[68, 77]		330		128.22
Наблюдаемое значение χ²: 3233.2657
Критическое значение χ²: 9.4877
Нулевая гипотеза отвергнута: распределение не нормальное

Проверка для средних возрастов:

Интервалы и частоты:
Интервал		Эмпир. частота	Теор. частота
[31, 32)		1		0.64
[32, 33)		0		1.85
[33, 34)		5		4.51
[34, 35)		11		7.57
[35, 36)		7		8.75
[36, 37)		6		6.97
[37, 38)		4		3.82
[38, 39)		1		1.44
[39, 40]		1		0.45
Наблюдаемое значение χ²: 4.9578
Критическое значение χ²: 12.5916
Нет оснований отвергнуть гипотезу: распределение нормальное


Критерий Пирсона показывает, что распределение возраста не нормальное (χ²_набл > χ²_кр), в то время как средние возрасты ближе к нормальному благодаря ЦПТ.

### Проверка гипотез о дисперсиях

По двум выборкам, сгенерированным из совокупности значений возраста, проверить нулевую гипотезу о равенстве дисперсий генеральных совокупностей при уровне значимости 0,05 при конкурирующей гипотезе

a) 𝐻1: 𝐷1 > 𝐷2

б) 𝐻1: 𝐷1 ≠ 𝐷2

In [7]:
def sample_variance(data):
    n = len(data)
    m = mean_calc(data)
    return sum((x - m) ** 2 for x in data) / (n - 1)

gamma = 0.95
delta = 3
t = 1.96
sigma = std_dev_calc(data)
n_sample = math.ceil((t**2 * sigma**2) / delta**2)

np.random.seed(42)  # Для воспроизводимости
samples = []
for _ in range(2):
    sample = np.random.choice(a=data, size=n_sample, replace=True)
    samples.append(sample)

x1, x2 = samples
alpha = 0.05

print("\n=== Проверка гипотезы о равенстве дисперсий ===")
s1_sq = sample_variance(x1)
s2_sq = sample_variance(x2)
F = max(s1_sq, s2_sq) / min(s1_sq, s2_sq)
k1 = len(x1) - 1
k2 = len(x2) - 1
print(f"Наблюдаемое значение F_набл = {F:.4f}")

# a) H1: D1 > D2 (правосторонняя)
F_crit_right_a = stats.f.ppf(1 - alpha, k1, k2)
print(f"a) H1: D1 > D2, F_кр = {F_crit_right_a:.4f}")
if F > F_crit_right_a:
    print("Отвергаем H0 в пользу H1: D1 > D2")
else:
    print("Нет оснований отвергать H0")

# б) H1: D1 ≠ D2 (двусторонняя)
F_crit_right_b = stats.f.ppf(1 - alpha / 2, k1, k2)
print(f"б) H1: D1 ≠ D2, F_кр = {F_crit_right_b:.4f}")
if F > F_crit_right_b:
    print("Отвергаем H0 в пользу H1: D1 ≠ D2")
else:
    print("Нет оснований отвергать H0")


=== Проверка гипотезы о равенстве дисперсий ===
Наблюдаемое значение F_набл = 1.1162
a) H1: D1 > D2, F_кр = 1.5288
Нет оснований отвергать H0
б) H1: D1 ≠ D2, F_кр = 1.6597
Нет оснований отвергать H0


**Вывод:** Проверка показывает, что дисперсии выборок равны для обеих альтернатив (F_набл < F_кр).

Взяв одну из выборок, при уровне значимости 0,05 проверить нулевую гипотезу о равенстве генеральной дисперсии значению, полученному в л/р № 1, при конкурирующей гипотезе:

a) 𝐻1: 𝐷(𝑥) > 𝜎2

б) 𝐻1: 𝐷(𝑥) ≠ 𝜎2

в) 𝐻1: 𝐷(𝑥) < 𝜎2

In [8]:
print("\n=== Проверка гипотезы о равенстве дисперсии с σ² ===")
sigma_sq = 144.92
sample_var = sample_variance(x1)
chi2 = (n_sample - 1) * sample_var / sigma_sq
print(f"χ² = {chi2:.4f}")

# a) H1: D(x) > σ² (правосторонняя)
chi2_crit_right = stats.chi2.ppf(1 - alpha, n_sample - 1)
print(f"\na)Конкурирующая гипотеза H1: D(x) > σ², χ²_кр = {chi2_crit_right:.4f}")
if chi2 > chi2_crit_right:
    print("Отвергаем H0 в пользу H1: D(x) > σ²")
else:
    print("Нет оснований отвергать H0")

# б) H1: D(x) ≠ σ² (двусторонняя)
chi2_left = stats.chi2.ppf(alpha / 2, n_sample - 1)
chi2_right = stats.chi2.ppf(1 - alpha / 2, n_sample - 1)
print(f"\nб) Конкурирующая гипотеза H1: D(x) ≠ σ², χ²_кр (левый) = {chi2_left:.4f}, χ²_кр (правый) = {chi2_right:.4f}")
if chi2 < chi2_left or chi2 > chi2_right:
    print("Отвергаем H0 в пользу H1: D(x) ≠ σ²")
else:
    print("Нет оснований отвергать H0")

# в) H1: D(x) < σ² (левосторонняя)
chi2_crit_left = stats.chi2.ppf(alpha, n_sample - 1)
print(f"\nв) Конкурирующая гипотеза H1: D(x) < σ², χ²_кр = {chi2_crit_left:.4f}")
if chi2 < chi2_crit_left:
    print("Отвергаем H0 в пользу H1: D(x) < σ²")
else:
    print("Нет оснований отвергать H0")


=== Проверка гипотезы о равенстве дисперсии с σ² ===
χ² = 60.0274

a)Конкурирующая гипотеза H1: D(x) > σ², χ²_кр = 80.2321
Нет оснований отвергать H0

б) Конкурирующая гипотеза H1: D(x) ≠ σ², χ²_кр (левый) = 41.3031, χ²_кр (правый) = 84.4764
Нет оснований отвергать H0

в) Конкурирующая гипотеза H1: D(x) < σ², χ²_кр = 44.0379
Нет оснований отвергать H0


**Вывод:** Для равенства σ²=144.92 с χ²=60.0274 нет оснований отвергать H0 ни при H1: D(x) > σ², ни при H1: D(x) ≠ σ², ни при H1: D(x) < σ², что указывает на соответствие дисперсии выборки генеральной дисперсии.